# Lecture 2.5 — Structured Outputs with Pydantic Models and TypedDict

**OpenAI Agents SDK — Complete Course | Section 02 — Agents: Configuration & Behaviour**

---

In this notebook we move beyond plain-text agent responses. By the end, you will be able to make an agent return a fully typed, validated Python object — a Pydantic `BaseModel` instance, a `TypedDict`, or a `dataclass` — instead of a raw string. This is the foundation for building agents whose output can be reliably consumed by downstream code.

**What we cover:**
- Why `output_type` exists and what it does under the hood
- Pydantic `BaseModel` — the most common and robust pattern
- `TypedDict` — lightweight structured output without Pydantic validation
- `dataclass` with `AgentOutputSchema(strict_json_schema=False)` — the escape hatch
- Nested Pydantic models for complex extraction tasks
- When to use which type (decision table)

## Cell 1 — Install the OpenAI Agents SDK

📌 **Notebook update notice:** the code shown in this lecture's video pins `openai-agents==0.17.4`. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version shown in the video. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one visible in the recording.

This cell pins the SDK to a specific version so every code cell in this notebook runs exactly as shown, regardless of what the SDK has moved on to since. If you want to try the latest release instead, remove the version pin and run

In [1]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.4 as shown in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00


## Cell 2 — API Key Setup

This notebook uses the **Google Colab Secrets** method to load your OpenAI API key. This is the safest approach — your key is never stored in the notebook file itself.

**Step-by-step instructions for Colab:**
1. Click the **🔑 key icon** in the left sidebar to open the Secrets panel.
2. Click **+ Add new secret**.
3. Set the name to exactly `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the value.
5. Toggle **Notebook access** to ON for this notebook.
6. Run the cell below.

**Running locally?** Skip the Colab steps above. Instead, set the environment variable in your terminal before launching Jupyter:

```bash
export OPENAI_API_KEY="your-key-here"
```

Then comment out the `userdata` lines below and remove the `os.environ` assignment (the SDK will pick up the key automatically from the environment).

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Model Name Variable

Every `Agent` definition in this notebook uses the `MODEL_NAME` variable declared here. Changing the value once in this cell updates the model used across the entire notebook — no need to hunt down individual `Agent` definitions.

You can find the full list of available OpenAI models at:
👉 https://platform.openai.com/docs/models

The default below, `gpt-5.4-mini`, is the current SDK default model and offers an excellent balance of speed, cost, and capability for structured-output tasks.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

Here is everything we need for this notebook:

| Import | Why we need it |
|---|---|
| `dataclass` from `dataclasses` | Decorator to define a dataclass output type |
| `TypedDict` from `typing_extensions` | Lightweight typed dict — no Pydantic required |
| `BaseModel` from `pydantic` | The most robust and common structured output type |
| `Reasoning` from `openai.types.shared` | Used in `ModelSettings` to control reasoning effort |
| `Agent` from `agents` | The core agent class |
| `AgentOutputSchema` from `agents` | Wraps an output type with explicit control over strict mode — our escape hatch for schemas that can't satisfy strict JSON schema constraints |
| `ModelSettings` from `agents` | Fine-grained model configuration (temperature, reasoning, etc.) |
| `Runner` from `agents` | Executes agent runs asynchronously |

**Note on `AgentOutputSchema`:** In most cases you pass your type directly as `output_type=MyModel` and the SDK wraps it internally. We import `AgentOutputSchema` explicitly for the cell where we need to disable strict mode — a capability only available when you construct the schema wrapper yourself.

In [4]:
from dataclasses import dataclass
from typing_extensions import TypedDict
from pydantic import BaseModel
from openai.types.shared import Reasoning
from agents import Agent, AgentOutputSchema, ModelSettings, Runner

## Cell 5 — Why Structured Outputs? (Concept)

By default, `final_output` on a `RunResult` is always a plain Python `str`. That is fine for conversational responses you will display to a user — but it creates friction the moment your code needs to do something with the output:

- Parse a JSON string manually
- Extract fields with regex or string slicing
- Hope the model formatted the output correctly

The `output_type` parameter removes all of that friction.

### What `output_type` does

When you set `output_type` on an `Agent`, the SDK:
1. Generates a strict JSON schema from your type (using Pydantic's `TypeAdapter`).
2. Sends that schema to the model and instructs it to use **structured outputs mode** — the model is required to produce JSON that exactly matches your schema.
3. Parses and validates the model's JSON response back into a typed Python object.
4. Returns that object as `result.final_output` — **not a string**.

### Strict mode (the default)

Internally the SDK wraps your type in `AgentOutputSchema(type, strict_json_schema=True)`. Strict mode is strongly recommended: it guarantees the model produces valid JSON that conforms exactly to your schema. The tradeoff is that some schema shapes are not allowed in strict mode — for example, `dict[int, str]` or optional fields without defaults.

### The escape hatch

If your type cannot satisfy strict mode, you can bypass it explicitly:

```python
output_type=AgentOutputSchema(MyType, strict_json_schema=False)
```

Without strict mode the model _may_ occasionally produce invalid JSON, so treat this as a last resort.

### Accessing typed output

- **Pydantic models** → dot notation: `result.final_output.my_field`
- **TypedDict** → dict key notation: `result.final_output["my_field"]`
- **Dataclasses** → dot notation: `result.final_output.my_field`
- **Convenience cast** → `result.final_output_as(MyModel)` — useful for type-checker satisfaction when `final_output` is typed as `Any`

## Cell 6 — Pydantic BaseModel: The Most Common Pattern

Pydantic `BaseModel` is the first-choice type for structured outputs. It gives you:
- **Automatic field validation** at parse time — the SDK raises a `ModelBehaviorError` if the model returns invalid data
- **Dot-notation access** to fields
- **IDE autocompletion and type checking**
- **Full compatibility with strict JSON schema mode**

In this cell we define a `NewsArticleSummary` model with four fields, wire it up as the agent's `output_type`, run a short article through the agent, and inspect the typed result.

**What to watch for in the output:**
- `type(result.final_output)` confirms this is a `NewsArticleSummary` instance, not a string.
- Field access uses dot notation: `result.final_output.headline`.
- `key_facts` is a `list[str]` — the SDK validated and parsed it correctly.

In [5]:
class NewsArticleSummary(BaseModel):
    headline: str
    summary: str
    key_facts: list[str]
    sentiment: str


agent = Agent(
    name="News Summariser",
    instructions=(
        "You summarise news articles. "
        "Extract the headline, a 2-3 sentence summary, "
        "up to 3 key facts, and the overall sentiment."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=NewsArticleSummary,
)

article = (
    "OpenAI has announced a new series of GPT-5 models "
    "optimised for agentic workflows. The models feature "
    "improved tool use, lower latency for multi-step tasks, "
    "and native support for structured outputs. Developers "
    "can access them via the API starting today."
)

result = await Runner.run(agent, article)

print(type(result.final_output))
print(result.final_output)
print("Headline:", result.final_output.headline)
print("Sentiment:", result.final_output.sentiment)
print("Key facts:", result.final_output.key_facts)

<class '__main__.NewsArticleSummary'>
headline='OpenAI Announces GPT-5 Models Optimized for Agentic Workflows' summary='OpenAI has introduced a new series of GPT-5 models designed for agentic workflows. The models are built to improve tool use, reduce latency in multi-step tasks, and provide native support for structured outputs. Developers can begin accessing them through the API today.' key_facts=['Optimized for agentic workflows', 'Improved tool use and lower latency for multi-step tasks', 'Available via API starting today'] sentiment='Positive'
Headline: OpenAI Announces GPT-5 Models Optimized for Agentic Workflows
Sentiment: Positive
Key facts: ['Optimized for agentic workflows', 'Improved tool use and lower latency for multi-step tasks', 'Available via API starting today']


## Cell 7 — TypedDict: Lightweight Structured Output

`TypedDict` is a lightweight alternative to Pydantic `BaseModel`. It defines a typed dictionary — you get structure and type annotations without Pydantic's validation overhead or the need to import `BaseModel`.

**Trade-offs vs Pydantic `BaseModel`:**

| | Pydantic BaseModel | TypedDict |
|---|---|---|
| Runtime field validation | ✅ Yes | ❌ No |
| Dot notation access | ✅ Yes | ❌ No (use `["key"]`) |
| IDE type checking | ✅ Yes | ✅ Yes |
| Strict schema compatibility | ✅ Yes | ✅ Yes |
| Import required | `pydantic` | `typing_extensions` |

Use `TypedDict` when you want structure and type hints but don't need Pydantic's validation. Access fields with `result.final_output["key"]` — dict notation, not dot notation.

In this cell we build a product review analyser. After the run, note that `type(result.final_output)` shows a `dict` (the SDK wraps `TypedDict` in a `TypeAdapter` which returns a plain dict at runtime).

In [6]:
class ProductReview(TypedDict):
    product_name: str
    rating: int
    pros: list[str]
    cons: list[str]
    recommendation: str


review_agent = Agent(
    name="Review Analyser",
    instructions=(
        "You analyse product reviews. "
        "Extract the product name, a rating out of 10, "
        "up to 3 pros, up to 3 cons, and a one-sentence "
        "recommendation."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=ProductReview,
)

review = (
    "I've been using the AeroPress coffee maker for 6 months. "
    "It makes excellent espresso-style coffee, is very portable, "
    "and easy to clean. The only downside is it only makes one "
    "cup at a time, which is annoying when having guests."
)

result = await Runner.run(review_agent, review)

print(type(result.final_output))
print(result.final_output)
print("Rating:", result.final_output["rating"])
print("Pros:", result.final_output["pros"])

<class 'dict'>
{'product_name': 'AeroPress coffee maker', 'rating': 8, 'pros': ['Makes excellent espresso-style coffee', 'Very portable', 'Easy to clean'], 'cons': ['Only makes one cup at a time'], 'recommendation': 'Great for solo coffee drinkers who want portable, easy-to-clean coffee with strong flavor, but less ideal for serving guests.'}
Rating: 8
Pros: ['Makes excellent espresso-style coffee', 'Very portable', 'Easy to clean']


## Cell 8 — Dataclass with AgentOutputSchema (Non-Strict Mode)

Before running this cell, pause and understand why it looks different from Cells 6 and 7.

### What normally happens with output_type

Every `output_type` you pass — Pydantic, TypedDict, dataclass, anything — gets silently wrapped by the SDK:

```python
AgentOutputSchema(YourType, strict_json_schema=True)
```

Inside `AgentOutputSchema`, Pydantic’s `TypeAdapter` generates a JSON schema from your type. The SDK then runs that schema through strict mode enforcement, and sends it to the OpenAI API with `strict: True` — meaning the model’s token generation is constrained at the infrastructure level. It physically cannot produce output that violates your schema.

### Why dataclasses break this flow

`TypeAdapter` reads your dataclass and generates a valid JSON schema from it. But if any field has a default value, `TypeAdapter` correctly marks it as optional — leaving it **out of the `required` array**.

Strict mode’s rule: **every field must be in `required`**. No exceptions.

So the SDK calls `ensure_strict_json_schema()` on the generated schema, finds a field missing from `required`, and raises a `UserError` — before the agent ever runs.

Pydantic `BaseModel` avoids this because it was designed with strict mode in mind. Optional fields in a `BaseModel` are declared as `Optional[str] = None`, which generates a schema where the field is still in `required` but allows `null` as a value — satisfying strict mode. Dataclasses have no equivalent mechanism.

### The fix

You do the wrapping yourself and flip the flag:

```python
output_type=AgentOutputSchema(JobPosting, strict_json_schema=False)
```

The SDK sees it’s already an `AgentOutputSchema` and skips its own wrapping. `TypeAdapter` still generates the schema, but `ensure_strict_json_schema()` is never called — no `UserError`.

The schema is still sent to the API and the model is still instructed to follow it. But there is no machine-level enforcement. The model tries its best and will almost always return the right structure — but failure is unlikely, not impossible. Under strict mode, failure is impossible.

### The full picture

| Type | Wrapped by | `strict_json_schema` | Result |
|---|---|---|---|
| Pydantic `BaseModel` | SDK silently | `True` | ✅ Works, machine-level guarantee |
| `TypedDict` | SDK silently | `True` | ✅ Works, machine-level guarantee |
| `dataclass` | SDK silently | `True` | ❌ `UserError` |
| `dataclass` | You explicitly | `False` | ✅ Works, no machine-level guarantee |

In [7]:
@dataclass
class JobPosting:
    job_title: str
    company: str
    required_skills: list[str]
    salary_range: str
    location: str


job_agent = Agent(
    name="Job Extractor",
    instructions=(
        "You extract structured job posting information "
        "from job descriptions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=AgentOutputSchema(
        JobPosting,
        strict_json_schema=False,
    ),
)

job_description = (
    "We are looking for a Senior Python Engineer to join "
    "our AI platform team in Bangalore. The role offers a "
    "salary range of 25-40 LPA. Required skills: Python, "
    "FastAPI, PostgreSQL, Docker, and experience with LLMs."
)

result = await Runner.run(job_agent, job_description)

print(type(result.final_output))
print(result.final_output)
print("Job title:", result.final_output.job_title)

<class '__main__.JobPosting'>
JobPosting(job_title='Senior Python Engineer', company='AI platform team', required_skills=['Python', 'FastAPI', 'PostgreSQL', 'Docker', 'LLMs'], salary_range='25-40 LPA', location='Bangalore')
Job title: Senior Python Engineer


## Cell 9 — Nested Pydantic Models

Pydantic `BaseModel` supports full nesting — you can define a model that contains fields typed as other models, or as lists of models. The SDK recursively generates the complete JSON schema for every level of nesting.

This makes nested models the best pattern for complex extraction tasks. Extracting structured data from a research paper, a legal document, or a product catalogue becomes straightforward:

- Define an outer model for the top-level structure
- Define inner models for nested entities (authors, line items, sections)
- Pass the outer model as `output_type` — the SDK handles the rest

In this cell we define an `Author` model nested inside a `ResearchPaper` model, then extract information from a plain-text description of a famous paper. Notice that iterating over `result.final_output.authors` gives us properly typed `Author` instances with dot-notation access.

In [8]:
class Author(BaseModel):
    name: str
    role: str


class ResearchPaper(BaseModel):
    title: str
    authors: list[Author]
    abstract: str
    key_contributions: list[str]
    publication_year: int


paper_agent = Agent(
    name="Paper Extractor",
    instructions=(
        "You extract structured information from academic "
        "paper descriptions."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=ResearchPaper,
)

paper_text = (
    "Attention Is All You Need was published in 2017 by "
    "Ashish Vaswani (Research Scientist) and Noam Shazeer "
    "(Senior Staff Engineer) at Google Brain. The paper "
    "introduced the Transformer architecture, eliminated "
    "recurrence in sequence models, and demonstrated that "
    "attention mechanisms alone are sufficient for "
    "state-of-the-art results in machine translation."
)

result = await Runner.run(paper_agent, paper_text)

print(result.final_output.title)
print(result.final_output.publication_year)
for author in result.final_output.authors:
    print(f"  {author.name} — {author.role}")

Attention Is All You Need
2017
  Ashish Vaswani — Research Scientist
  Noam Shazeer — Senior Staff Engineer


## Cell 10 — What `output_type` Does NOT Do

Before moving on, a few important clarifications about the limits of structured outputs:

**It validates structure, not correctness.**  
If the model returns a `NewsArticleSummary` with a `sentiment` field that says `"positive"` when the article is actually negative, the SDK will not catch that. Structured outputs guarantee the *shape* of the data, not the accuracy of the content.

**It is all-or-nothing.**  
When `output_type` is set, the model cannot mix structured output with prose. It must return a JSON object that matches the schema — nothing else. There is no mode where the model produces a structured object and also adds a natural-language explanation.

**In multi-agent handoff chains, only the last agent's output is validated.**  
If Agent A hands off to Agent B, and only Agent B has `output_type` set, the final `result.final_output` will be Agent B's typed output. We will revisit this explicitly in Section 5 when we cover multi-agent orchestration.

**`output_type` forces structured outputs mode entirely.**  
There is no partial structured output — once you set `output_type`, the model is operating in structured outputs mode for the entire run.

## Cell 11 — When to Use Which Type

Use this table as a quick reference when deciding how to type your agent's output:

| Scenario | Recommended type |
|---|---|
| Output consumed by downstream code | `Pydantic BaseModel` (first choice) |
| Lightweight structure, no validation needed | `TypedDict` |
| Complex nested data extraction | Nested `Pydantic BaseModel` |
| Schema that cannot satisfy strict mode | `AgentOutputSchema(Type, strict_json_schema=False)` |
| Output displayed to a human (no code consumes it) | Plain `str` — no `output_type` needed |
| Output piped into another agent | `Pydantic BaseModel` |

**Rule of thumb:** Default to Pydantic `BaseModel`. Drop to `TypedDict` when you want to avoid the Pydantic dependency. Use `AgentOutputSchema` with `strict_json_schema=False` only when strict mode blocks you. Leave `output_type` unset when the output is for human consumption.